# MedSigLIP-448: Register, Deploy, Embed

Registers `google/medsiglip-448` as a dual-encoder CustomModel in the Snowflake Model Registry, deploys it as a GPU inference service on SPCS, and embeds the DICOM corpus for text-to-image search.

**Prerequisites** (all created by `sql/admin/00_account_setup.sql` and `setup.sh`):
- `SF_CLINICAL_DB.EXPLORER` with `DICOM_FILE_MANIFEST` populated (2,848 rows from `02_stages.sql`)
- `DICOM_TO_BASE64_PNG` UDTF (from `03_udfs.sql`)
- `DICOM_GPU_POOL` compute pool (GPU_NV_S)
- `DICOM_EXPLORER_EAI` external access integration attached to this notebook
- HuggingFace token available as the `HF_TOKEN_SECRET` secret, or entered at the prompt

**The model is gated.** The HuggingFace account behind the token must have accepted the terms at https://huggingface.co/google/medsiglip-448 first. An unaccepted token fails with a 401 that looks exactly like an invalid token.

Run the cells in order. This notebook does not need a GPU itself - the model loads on CPU for registration. GPU is only needed for the SPCS service.

*Co-authored with CoCo*

In [ ]:
# Versions are pinned: SigLIP support in transformers moves, and an
# unpinned upgrade can silently change embedding values.
!pip install --quiet \
    "transformers==4.44.2" \
    "torch==2.4.1" \
    "pillow>=10.0.0" \
    "huggingface_hub>=0.24.0" \
    "sentencepiece" \
    "protobuf" \
    "accelerate"

## Configuration
Every object name is declared once here.

In [ ]:
DEMO_DB       = "SF_CLINICAL_DB"
DEMO_SCHEMA   = "EXPLORER"
GPU_POOL      = "DICOM_GPU_POOL"
MODEL_NAME    = "MEDSIGLIP_448"
MODEL_VERSION = "V2"
SERVICE_NAME  = "MEDSIGLIP_448_SVC"
HF_REPO       = "google/medsiglip-448"
HF_SECRET     = "HF_TOKEN_SECRET"
EMBED_DIM     = 1152

FQ = f"{DEMO_DB}.{DEMO_SCHEMA}"

from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql(f"USE SCHEMA {FQ}").collect()

print(f"Session ready. Schema: {FQ}")
print(f"Role: {session.get_current_role()}  Warehouse: {session.get_current_warehouse()}")

## HuggingFace token
Resolved in three steps, in order:

1. **In-process** via `_snowflake.get_generic_secret_string`. Works in UDFs and stored procedures, but the notebook container runtime does not expose the `_snowflake` module, so this normally fails here.
2. **Short-lived reader UDF.** This is the path that actually works in a notebook. The UDF is dropped immediately after the read - while it exists, anyone with USAGE on it could call it and retrieve the token, so it must not be left behind.
3. **Interactive prompt**, only when stdin is a TTY. Under `EXECUTE NOTEBOOK` there is no stdin, and an unguarded `getpass` raises a bare `EOFError` that tells you nothing. Step 3 is skipped in that case and a diagnostic error is raised instead.

The token is never written to disk or printed.

In [ ]:
import sys

HF_TOKEN = None
_secret_errors = []

# Path 1: in-process secret read. Works in UDFs and procedures; the notebook
# container runtime does NOT expose the _snowflake module, so this usually
# fails here and we fall through to path 2.
try:
    import _snowflake
    HF_TOKEN = _snowflake.get_generic_secret_string("hf_token")
    if HF_TOKEN:
        print("Token loaded in-process via _snowflake.")
except Exception as e:
    _secret_errors.append(f"_snowflake: {type(e).__name__}: {e}")

# Path 2: read the secret through a short-lived UDF.
# This is the path that actually works in a notebook. The UDF is dropped
# immediately in the finally block - while it exists, anyone with USAGE on it
# could call it and get the token back, so it must not be left behind.
if not HF_TOKEN:
    probe = f"{FQ}._HF_TOKEN_READER"
    try:
        session.sql(f"""
            CREATE OR REPLACE FUNCTION {probe}()
            RETURNS STRING
            LANGUAGE PYTHON
            RUNTIME_VERSION = '3.11'
            HANDLER = 'read_token'
            EXTERNAL_ACCESS_INTEGRATIONS = (DICOM_EXPLORER_EAI)
            SECRETS = ('hf_token' = {FQ}.{HF_SECRET})
            AS $$
import _snowflake
def read_token():
    return _snowflake.get_generic_secret_string('hf_token')
            $$
        """).collect()
        HF_TOKEN = session.sql(f"SELECT {probe}() AS T").collect()[0]["T"]
        if HF_TOKEN:
            print("Token loaded via short-lived reader UDF.")
    except Exception as e:
        _secret_errors.append(f"reader UDF: {type(e).__name__}: {e}")
    finally:
        try:
            session.sql(f"DROP FUNCTION IF EXISTS {probe}()").collect()
        except Exception:
            pass

# Path 3: interactive prompt. Only viable in a live session - under
# EXECUTE NOTEBOOK there is no stdin and getpass raises a bare EOFError,
# which is useless for diagnosis. Guard on isatty and fail with something
# actionable instead.
if not HF_TOKEN:
    if sys.stdin is not None and sys.stdin.isatty():
        import getpass
        HF_TOKEN = getpass.getpass("HuggingFace token: ").strip()
    else:
        raise RuntimeError(
            "Could not read the HuggingFace token, and this session is not "
            "interactive so it cannot be prompted for.\n\n"
            "Attempts:\n  - " + "\n  - ".join(_secret_errors) + "\n\n"
            f"Check that the notebook has the secret attached:\n"
            f"  SECRETS = ('hf_token' = {FQ}.{HF_SECRET})\n"
            f"  EXTERNAL_ACCESS_INTEGRATIONS = (DICOM_EXPLORER_EAI)\n"
            f"and that the EAI allows it:\n"
            f"  DESCRIBE EXTERNAL ACCESS INTEGRATION DICOM_EXPLORER_EAI;\n"
            f"  -> ALLOWED_AUTHENTICATION_SECRETS must list {FQ}.{HF_SECRET}"
        )

assert HF_TOKEN, "A HuggingFace token is required to download the gated model."
print(f"Token present ({len(HF_TOKEN)} chars).")

In [ ]:
import os
import time

# Force the classic cdn-lfs transfer path instead of Xet.
#
# hf-xet is preinstalled in the container runtime, so huggingface_hub prefers
# Xet storage. Xet fans out to hosts like cas-bridge.xethub.hf.co and
# us.aws.cdn.hf.co, which rotate and are not documented. If any of them is
# blocked by your egress rule the download does NOT error - it retries
# silently and the cell hangs with no output, which is indistinguishable from
# a slow download. Measured for reference: with egress open this download
# completes in ~75 seconds.
#
# Must be set BEFORE importing huggingface_hub - the flag is read at import.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")

from huggingface_hub import snapshot_download

model_dir = "/tmp/medsiglip-448"
os.makedirs(model_dir, exist_ok=True)

print(f"Downloading {HF_REPO} (Xet disabled) ...", flush=True)
_start = time.time()

try:
    snapshot_path = snapshot_download(
        repo_id=HF_REPO,
        cache_dir=model_dir,
        token=HF_TOKEN,
    )
except Exception as e:
    msg = str(e)
    if "NameResolutionError" in msg or "Max retries exceeded" in msg:
        raise RuntimeError(
            "A HuggingFace host could not be resolved, which means your "
            "external access integration does not allow it.\n\n"
            "HuggingFace serves weights from dynamic CDN hostnames, so an "
            "allowlist is hard to maintain. Widen the egress rule:\n"
            "  ALTER NETWORK RULE SF_CLINICAL_DB.EXPLORER.DICOM_EXPLORER_EGRESS_RULE\n"
            "    SET VALUE_LIST = ('0.0.0.0:443', '0.0.0.0:80');\n\n"
            f"Original error: {msg}"
        ) from e
    if "401" in msg or "gated" in msg.lower() or "awaiting" in msg.lower():
        raise RuntimeError(
            f"HuggingFace rejected the request for {HF_REPO}.\n"
            f"This model is GATED. Accept the terms at "
            f"https://huggingface.co/{HF_REPO} using the same account that "
            f"issued this token, then re-run.\nOriginal error: {msg}"
        ) from e
    raise

print(f"Model downloaded in {time.time() - _start:.0f}s to: {snapshot_path}")
print(f"Contents: {sorted(os.listdir(snapshot_path))}")

## Dual-encoder CustomModel
Two inference endpoints sharing one vector space:
- `predict(IMAGE_BYTES)` - base64 PNG to a 1152-dim image embedding
- `embed_text(TEXT)` - natural language to a 1152-dim text embedding

`embed_text` uses `padding="max_length"`. This is not cosmetic: SigLIP was trained with a fixed 64-token padded context, and `padding=True` (pad to longest in batch) produces different, degraded text embeddings with no error raised. Query vectors would drift out of alignment with the image vectors and retrieval quality would quietly drop.

In [ ]:
import base64
from io import BytesIO

import numpy as np
import pandas as pd
from snowflake.ml.model import custom_model


class MedSigLIPModel(custom_model.CustomModel):
    """google/medsiglip-448 dual encoder for medical image and text embeddings."""

    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        import torch
        from transformers import AutoModel, AutoProcessor

        model_path = self.context.path("model_dir")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = AutoModel.from_pretrained(model_path).to(self.device).eval()
        self.processor = AutoProcessor.from_pretrained(model_path)

    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        """IMAGE_BYTES (base64 PNG) -> EMBEDDING (list of 1152 floats)."""
        import torch
        from PIL import Image

        embeddings = []
        for _, row in input_df.iterrows():
            image = Image.open(BytesIO(base64.b64decode(row["IMAGE_BYTES"]))).convert("RGB")
            inputs = self.processor(images=image, return_tensors="pt").to(self.device)
            with torch.no_grad():
                feats = self.model.get_image_features(**inputs)
            embeddings.append(feats.cpu().numpy().flatten().tolist())

        return pd.DataFrame({"EMBEDDING": embeddings})

    @custom_model.inference_api
    def embed_text(self, input_df: pd.DataFrame) -> pd.DataFrame:
        """TEXT -> EMBEDDING (list of 1152 floats), same space as predict().

        padding="max_length" is required. SigLIP trains with a fixed 64-token
        padded context; padding to the longest sequence in the batch instead
        yields silently different embeddings.
        """
        import torch

        embeddings = []
        for _, row in input_df.iterrows():
            inputs = self.processor(
                text=row["TEXT"],
                return_tensors="pt",
                padding="max_length",
            ).to(self.device)
            with torch.no_grad():
                feats = self.model.get_text_features(**inputs)
            embeddings.append(feats.cpu().numpy().flatten().tolist())

        return pd.DataFrame({"EMBEDDING": embeddings})


print("MedSigLIPModel defined: predict(IMAGE_BYTES), embed_text(TEXT)")

In [ ]:
from PIL import Image

# Synthetic 448x448 gray frame, enough to exercise both encoders.
_buf = BytesIO()
Image.new("RGB", (448, 448), color=(128, 128, 128)).save(_buf, format="PNG")
sample_img_b64 = base64.b64encode(_buf.getvalue()).decode("utf-8")

sample_image_df = pd.DataFrame({"IMAGE_BYTES": [sample_img_b64]})
sample_text_df  = pd.DataFrame({"TEXT": ["cardiac CT angiography with calcification"]})

medsiglip = MedSigLIPModel(custom_model.ModelContext(model_dir=snapshot_path))
print(f"Loaded on device: {medsiglip.device}")

image_out = medsiglip.predict(sample_image_df)
text_out  = medsiglip.embed_text(sample_text_df)

img_dim = len(image_out["EMBEDDING"][0])
txt_dim = len(text_out["EMBEDDING"][0])
print(f"Image embedding: {img_dim} dims")
print(f"Text embedding:  {txt_dim} dims")

assert img_dim == EMBED_DIM, f"Expected {EMBED_DIM}, got {img_dim}"
assert txt_dim == EMBED_DIM, f"Expected {EMBED_DIM}, got {txt_dim}"

a = np.array(image_out["EMBEDDING"][0])
b = np.array(text_out["EMBEDDING"][0])
print(f"Cosine(gray image, cardiac text): {np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)):.4f}")
print("Low similarity is expected for a blank frame - confirms the encoders are distinct.")

## Register in the Model Registry
`target_platforms` is set to SPCS only. Without it the registry also resolves a warehouse-compatible environment, which fails for a GPU torch model.

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import model_signature

reg = Registry(session=session, database_name=DEMO_DB, schema_name=DEMO_SCHEMA)

# Drop service before model - a model backing a live service cannot be dropped.
# Deliberately not wrapped in try/except: a silent failure here surfaces later
# as a much less obvious log_model error.
session.sql(f"DROP SERVICE IF EXISTS {FQ}.{SERVICE_NAME}").collect()
session.sql(f"DROP MODEL IF EXISTS {FQ}.{MODEL_NAME}").collect()
print("Cleared any existing service and model.")

signatures = {
    "predict":    model_signature.infer_signature(sample_image_df, image_out),
    "embed_text": model_signature.infer_signature(sample_text_df,  text_out),
}

model_version = reg.log_model(
    model=medsiglip,
    model_name=MODEL_NAME,
    version_name=MODEL_VERSION,
    signatures=signatures,
    target_platforms=["SNOWPARK_CONTAINER_SERVICES"],
    # cuda_version is REQUIRED to build a GPU-capable model image. Without it,
    # create_service(gpu_requests="1") fails with:
    #   "model ... does not have GPU runtime support"
    # 12.1 matches the CUDA build of the pinned torch==2.4.1 wheel.
    options={"cuda_version": "12.1"},
    pip_requirements=[
        "transformers==4.44.2",
        "torch==2.4.1",
        "pillow>=10.0.0",
        "huggingface_hub>=0.24.0",
        "sentencepiece",
        "protobuf",
        "accelerate",
    ],
    comment="MedSigLIP-448 dual encoder: predict(IMAGE_BYTES)->embedding, embed_text(TEXT)->embedding",
)

print(f"Logged {model_version.model_name} / {model_version.version_name}")

In [ ]:
# Builds a container image and starts the service. Expect 10-20 minutes on a
# cold GPU pool - most of that is the image build, not the model.
model_version.create_service(
    service_name=SERVICE_NAME,
    service_compute_pool=GPU_POOL,
    gpu_requests="1",
    max_instances=1,
)

print(f"Service {SERVICE_NAME} deployment initiated on {GPU_POOL}.")

In [ ]:
import json
import time

# Poll until READY. create_service can return before the container is serving.
deadline = time.time() + 45 * 60
status = None
while time.time() < deadline:
    raw = session.sql(f"SELECT SYSTEM$GET_SERVICE_STATUS('{FQ}.{SERVICE_NAME}')").collect()[0][0]
    try:
        entries = json.loads(raw)
        status = entries[0].get("status") if isinstance(entries, list) else str(entries)
    except (json.JSONDecodeError, TypeError, IndexError):
        status = str(raw)

    print(f"{time.strftime('%H:%M:%S')}  {status}")
    if status and status.upper() in ("READY", "RUNNING"):
        break
    if status and status.upper() in ("FAILED", "DONE"):
        raise RuntimeError(f"Service entered terminal state {status}. Check container logs.")
    time.sleep(30)

print(f"\nFinal status: {status}")

In [ ]:
-- Smoke test the text endpoint from SQL.
-- ARRAY_SIZE on the returned VARIANT confirms 1152 dims end to end.
SELECT ARRAY_SIZE(
    MEDSIGLIP_448_SVC!embed_text('cardiac CT angiography with contrast'):EMBEDDING
) AS TEXT_DIMS;

## Phase 1: DICOM to base64 PNG
Uses the `DICOM_TO_BASE64_PNG` UDTF from `sql/core/03_udfs.sql`, driven by `DICOM_FILE_MANIFEST` and `DICOM_SERIES_REGISTRY` from `sql/core/02_stages.sql`. The warehouse parallelizes the conversion.

The UDTF applies rescale slope/intercept and a fixed window mapping, matching `RENDER_DICOM_SLICE`, so the pixels embedded here are the same pixels the app renders.

In [ ]:
-- Render base64 PNGs by calling the same procedure TASK_DICOM_BASE64 calls.
-- This replaced a CREATE OR REPLACE TABLE ... AS SELECT, which rebuilt every row
-- on every run. The procedure only renders files that have no image yet, and it
-- passes each source's window fallback from DICOM_SOURCE_CONFIG.
CALL SF_CLINICAL_DB.EXPLORER.SP_CONVERT_IMAGES(100000);


In [ ]:
-- TOTAL_ROWS must equal DISTINCT_FILES. Any gap means more than one image row
-- exists for a file, which would misalign embeddings downstream.
SELECT
    COUNT(*)                   AS TOTAL_ROWS,
    COUNT(DISTINCT FILE_NAME)  AS DISTINCT_FILES,
    COUNT(DISTINCT COLLECTION) AS COLLECTIONS,
    MIN(LENGTH(IMAGE_BYTES))   AS MIN_B64_LEN,
    MAX(LENGTH(IMAGE_BYTES))   AS MAX_B64_LEN
FROM SF_CLINICAL_DB.EXPLORER.DICOM_BASE64_IMAGES;


## Phase 2: Embed via the SPCS service
The service function is called directly in SQL, so each embedding stays on the same row as its `FILE_NAME` by construction.

This deliberately avoids the earlier pattern of running the model on a DataFrame and joining the result back on `IMAGE_BYTES`. Blank slices at series boundaries produce byte-identical base64, so that join fanned out and attached embeddings to the wrong files - silently, with no error.

In [ ]:
-- Embed by calling the same procedure TASK_DICOM_EMBED calls. It embeds only
-- images that have no vector yet, so re-running does not pay for GPU inference
-- twice, and it returns a SKIPPED message instead of failing if the service is
-- suspended.
CALL SF_CLINICAL_DB.EXPLORER.SP_EMBED_PENDING(100000);


In [ ]:
-- Correctness gates:
--   TOTAL_ROWS = DISTINCT_FILES  -> no fanout
--   EMB_TYPE   = VECTOR          -> cast landed
--   MAX_SELF_DIST = 0            -> vectors are well formed
SELECT
    COUNT(*)                            AS TOTAL_ROWS,
    COUNT(DISTINCT FILE_NAME)           AS DISTINCT_FILES,
    COUNT(DISTINCT SERIES_INSTANCE_UID) AS SERIES,
    COUNT(DISTINCT COLLECTION)          AS COLLECTIONS,
    ANY_VALUE(TYPEOF(EMBEDDING))        AS EMB_TYPE,
    MAX(VECTOR_L2_DISTANCE(EMBEDDING, EMBEDDING)) AS MAX_SELF_DIST
FROM SF_CLINICAL_DB.EXPLORER.DICOM_EMBEDDINGS;


## Text-to-image retrieval
Encodes a query with `embed_text` and ranks images by cosine similarity, all in SQL. This is the same path `SEARCH_DICOM_IMAGES` takes, minus the Cortex Search index.

A cardiac query should rank the cardiac CTA series above the NLST chest screening series. If the ranking looks arbitrary, suspect the `embed_text` padding mode before anything else.

In [ ]:
WITH q AS (
    SELECT MEDSIGLIP_448_SVC!embed_text(
        'cardiac CT angiography with calcified aortic valve'
    ):EMBEDDING::VECTOR(FLOAT, 1152) AS QVEC
)
SELECT
    e.LABEL,
    e.COLLECTION,
    COUNT(*) AS SLICES,
    ROUND(MAX(VECTOR_COSINE_SIMILARITY(e.EMBEDDING, q.QVEC)), 4) AS BEST_SIMILARITY
FROM SF_CLINICAL_DB.EXPLORER.DICOM_EMBEDDINGS e, q
GROUP BY e.LABEL, e.COLLECTION
ORDER BY BEST_SIMILARITY DESC;

## Next
1. `snow sql -f dicom-search/byo_search.sql` - Cortex Search BYO vector service and `SEARCH_DICOM_IMAGES`
2. `snow sql -f sql/core/09_agent_definition.sql` - agent with metadata and image search tools
3. `snow streamlit deploy dicom_explorer_sf --replace -p app` - the app

The GPU pool bills while active. Suspend it between demos:
```sql
ALTER SERVICE SF_CLINICAL_DB.EXPLORER.MEDSIGLIP_448_SVC SUSPEND;
ALTER COMPUTE POOL DICOM_GPU_POOL SUSPEND;
```
`DICOM_EMBEDDINGS` and the Cortex Search index persist independently, so text metadata search and the rest of the app keep working while the GPU is down. Only new embeddings and image-search queries need the service resumed.